<a href="https://colab.research.google.com/github/raheelarif86/AI_Training_November25/blob/main/OT_Risk_Assessment_2nd_Strive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================
# STEP 1: INSTALL REQUIRED LIBRARIES
# =====================================================

!pip install -q crewai crewai-tools gradio openai pandas python-dotenv


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 666.8/666.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.8/766.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.4/723.4 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 8.0 MB

In [9]:
# =====================================================
# STEP 2: LOAD OPENAI API KEY
# =====================================================


from google.colab import userdata
import os

# Fetch the API key named 'openai' from Colab Secrets and set it as OPENAI_API_KEY environment variable
os.environ["OPENAI_API_KEY"] = userdata.get("openai")

print("OpenAI API key loaded from Colab Secrets.")

OpenAI API key loaded from Colab Secrets.


In [10]:
# =====================================================
# STEP 3: IMPORTS
# =====================================================

import gradio as gr
import pandas as pd
from crewai import Agent, Task, Crew, LLM


In [11]:
# =====================================================
# STEP 4: GLOBAL ASSESSMENT CONTEXT
# =====================================================

ASSESSMENT_CONTEXT = """
Industry: Nuclear
Facility Type: Brownfield
Regulator: FANR (UAE)

Standards:
- FANR-RG-ICT
- FANR-REG-01 / REG-02
- NEI 08-09 (Cyber Security Plan)
- NEI 10-04 (Cyber Security Controls)
- NEI 13-10 (Cyber Security Assessments)

OT Architecture:
- Purdue Levels 0–4
- IT–OT interconnected
- Partial logical air-gap
- Mixed vendor environment

Threat Model:
- Nation-state
- Insider (malicious / negligent)
- Supply chain compromise
- Accidental misconfiguration

Constraints:
- No active scanning
- Documentation & interview-based only
- Sanitized, regulator-safe language required
"""


In [12]:
# =====================================================
# STEP 5: LLM CONFIGURATION (NUCLEAR SAFE)
# =====================================================

nuclear_llm = LLM(
    model="gpt-4.1",

    # Low temperature = deterministic, audit-safe
    temperature=0.2,

    # Conservative randomness
    top_p=0.9,

    # Large outputs (full reports)
    max_tokens=4096,

    # Reduce repetition
    frequency_penalty=0.1,

    # No creative drift
    presence_penalty=0.0
)


In [13]:
# =====================================================
# STEP 6: DEFINE AGENTS
# =====================================================

ot_security_expert = Agent(
    role="OT Cyber Security Expert (Nuclear)",
    goal="Identify OT cyber risks impacting nuclear safety and availability",
    backstory="Expert in nuclear OT systems and safety-critical environments",
    llm=nuclear_llm,
    verbose=True
)

compliance_expert = Agent(
    role="Nuclear Cyber Compliance Specialist",
    goal="Assess compliance against FANR and NEI requirements",
    backstory="Experienced in FANR audits and NEI nuclear guidance",
    llm=nuclear_llm,
    verbose=True
)

threat_analyst = Agent(
    role="OT Threat Intelligence Analyst",
    goal="Develop realistic, sanitized OT threat scenarios",
    backstory="Focuses on nuclear-relevant threats without exploit detail",
    llm=nuclear_llm,
    verbose=True
)

risk_analyst = Agent(
    role="OT Risk Analyst",
    goal="Prioritize risks using Likelihood × Impact methodology",
    backstory="Safety-first nuclear risk analysis specialist",
    llm=nuclear_llm,
    verbose=True
)

report_writer = Agent(
    role="Nuclear Cybersecurity Report Writer",
    goal="Produce regulator-ready cybersecurity assessment reports",
    backstory="Writes audit-safe documents for FANR and executives",
    llm=nuclear_llm,
    verbose=True
)


In [15]:
# =====================================================
# STEP 7: DEFINE TASKS
# =====================================================

threat_task = Task(
    description=f"""
Using the context below:
{ASSESSMENT_CONTEXT}

Develop sanitized OT cyber threat scenarios relevant to nuclear facilities.
Avoid exploit-level or tactical details.
""",
    expected_output="A detailed list of sanitized OT cyber threat scenarios relevant to nuclear facilities, avoiding exploit-level or tactical details.",
    agent=threat_analyst
)

compliance_task = Task(
    description="""
Map OT practices against:
- FANR-RG-ICT
- FANR-REG-01 / REG-02
- NEI 08-09, 10-04, 13-10

Identify compliance gaps using regulator-safe language.
""",
    expected_output="A comprehensive report detailing compliance gaps against FANR and NEI requirements, written in regulator-safe language.",
    agent=compliance_expert
)

risk_task = Task(
    description="""
Using threat scenarios and compliance gaps:
- Assign Likelihood (Low/Medium/High)
- Assign Impact (Safety, Availability, Regulatory)
- Prioritize risks qualitatively
""",
    expected_output="A risk register prioritizing identified risks based on Likelihood (Low/Medium/High) and Impact (Safety, Availability, Regulatory).",
    agent=risk_analyst
)

mitigation_task = Task(
    description="""
Recommend mitigations aligned with FANR and NEI expectations.
Focus on governance, architecture, and monitoring.
Avoid operational disruption.
""",
    expected_output="Actionable mitigation recommendations focused on governance, architecture, and monitoring, aligned with FANR and NEI expectations, without disrupting operations.",
    agent=ot_security_expert
)

report_task = Task(
    description="""
Compile a complete assessment report including:
- Executive Summary
- Threat Scenarios
- Compliance Gap Analysis
- Risk Register
- Mitigation Recommendations
- 30/60/90-Day Roadmap
- Technical Appendix

Ensure PDF-ready, regulator-safe language.
""",
    expected_output="A complete, regulator-ready cybersecurity assessment report in PDF format, including an Executive Summary, Threat Scenarios, Compliance Gap Analysis, Risk Register, Mitigation Recommendations, 30/60/90-Day Roadmap, and Technical Appendix.",
    agent=report_writer
)

In [16]:
# =====================================================
# STEP 8: CREATE CREW
# =====================================================

crew = Crew(
    agents=[
        threat_analyst,
        compliance_expert,
        risk_analyst,
        ot_security_expert,
        report_writer
    ],
    tasks=[
        threat_task,
        compliance_task,
        risk_task,
        mitigation_task,
        report_task
    ],
    process="sequential",
    verbose=True
)



In [17]:
# =====================================================
# STEP 9: RUN ASSESSMENT FUNCTION
# =====================================================

def run_nuclear_ot_assessment():
    """
    Executes CrewAI workflow and returns:
    1. Full assessment report
    2. Risk register CSV
    """

    report = crew.kickoff()

    risk_register = pd.DataFrame([
        {
            "Risk": "Unauthorized logical access to OT systems",
            "Likelihood": "Medium",
            "Impact": "High (Safety / Regulatory)",
            "Risk Rating": "High"
        },
        {
            "Risk": "Supply chain integrity weaknesses",
            "Likelihood": "Low",
            "Impact": "High",
            "Risk Rating": "Medium"
        }
    ])

    csv_path = "nuclear_ot_risk_register.csv"
    risk_register.to_csv(csv_path, index=False)

    return report, csv_path


In [18]:
# =====================================================
# STEP 10: GRADIO UI
# =====================================================

with gr.Blocks() as demo:
    gr.Markdown("## ⚛️ Nuclear OT Cyber Risk Assessment")
    gr.Markdown("""
    **Regulator:** FANR (UAE)
    **Standards:** FANR + NEI
    **Method:** Interview & documentation-based
    **Output:** Regulator-safe assessment
    """)

    run_button = gr.Button("Run Risk Assessment")
    report_output = gr.Textbox(
        label="Assessment Report (PDF-ready)",
        lines=25
    )
    csv_output = gr.File(
        label="Risk Register (CSV)"
    )

    run_button.click(
        fn=run_nuclear_ot_assessment,
        outputs=[report_output, csv_output]
    )

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ad182497db1e30b658.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
